# Limpeza de Dados: The Movies Dataset

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data_csv

--- 
## Carregando Dados

In [2]:
caminho = '../data/raw/movies_dataset/movies_metadata.csv'
df_movies = load_data_csv(caminho)

Dados carregados com sucesso! Formato: (45466, 24)


---
## Aplicando Limpeza de Missing Data 

In [3]:
# Dropping colunas irrelevantes para a análise que possuem alta cardinalidade de texto
# Essas colunas não são úteis para a análise e podem introduzir ruído
# Além disso, possuem 50% ou mais de valores ausentes, o que pode prejudicar a qualidade dos dados

colunas_para_dropar = ['homepage', 'tagline', 'overview','poster_path', 'status', 'imdb_id']

df_movies_limpo = df_movies.drop(columns=colunas_para_dropar)

In [12]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41190 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41190 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41190 non-null  str    
 3   genres                 41190 non-null  str    
 4   id                     41190 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41190 non-null  str    
 7   popularity             41190 non-null  str    
 8   production_companies   41190 non-null  str    
 9   production_countries   41190 non-null  str    
 10  release_date           41170 non-null  str    
 11  revenue                41190 non-null  float64
 12  runtime                41190 non-null  float64
 13  spoken_languages       41190 non-null  str    
 14  title                  41190 non-null  str    
 15  video             

In [4]:
# Dropando filmes com 0 votos
df_movies_limpo = df_movies_limpo[df_movies_limpo['vote_count'] > 0]

In [10]:
numero_filmes_zero_votos = (df_movies_limpo['vote_count'] == 0).sum()
print(f'Número de filmes com 0 votos: {numero_filmes_zero_votos}')

Número de filmes com 0 votos: 0


In [5]:
# Dropando coluna runtime
df_movies_limpo = df_movies_limpo.dropna(subset=['runtime'])
df_movies_limpo = df_movies_limpo[df_movies_limpo['runtime'] > 0]

In [11]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41190 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41190 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41190 non-null  str    
 3   genres                 41190 non-null  str    
 4   id                     41190 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41190 non-null  str    
 7   popularity             41190 non-null  str    
 8   production_companies   41190 non-null  str    
 9   production_countries   41190 non-null  str    
 10  release_date           41170 non-null  str    
 11  revenue                41190 non-null  float64
 12  runtime                41190 non-null  float64
 13  spoken_languages       41190 non-null  str    
 14  title                  41190 non-null  str    
 15  video             

In [ ]:
# Calculando os nulos de todas as colunas
nulos_por_coluna = df_movies.isnull().sum()

# Filtramndopara manter APENAS as colunas que têm buracos (> 0)
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]

# Calculamos a porcentagem
percentual_nulos = (colunas_com_nulos / len(df_movies)) * 100

# Juntando tudo num DataFrame de resumo
df_resumo_nulos = pd.DataFrame({
    'Valores Nulos (Qtd)': colunas_com_nulos,
    'Perda de Dados (%)': percentual_nulos
}).sort_values(by='Perda de Dados (%)', ascending=False)

# Selecionando apenas as colunas com perda de dados menor que 0.1% para considerar o drop
df_resumo_nulos_para_dropar = df_resumo_nulos[df_resumo_nulos['Perda de Dados (%)'] < 0.1]

In [ ]:
# Dropando as linhas com menos de 0.1% de perda de dados

# Criando uma lista de colunas que tem perda de dados menor ou igual a 0.1%
indices_limpos = [i.split(' (')[0] for i in df_resumo_nulos_para_dropar.index]

# Criando uma lista segura de colunas para drop, garantindo que elas existam no DataFrame
lista_segura = [coluna for coluna in indices_limpos if coluna in df_movies_limpo.columns]

# Dropando as linhas com menos de 0.1% de perda de dados
df_movies_limpo = df_movies_limpo.dropna(subset=lista_segura)

In [16]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41184 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41184 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41184 non-null  str    
 3   genres                 41184 non-null  str    
 4   id                     41184 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41184 non-null  str    
 7   popularity             41184 non-null  str    
 8   production_companies   41184 non-null  str    
 9   production_countries   41184 non-null  str    
 10  release_date           41164 non-null  str    
 11  revenue                41184 non-null  float64
 12  runtime                41184 non-null  float64
 13  spoken_languages       41184 non-null  str    
 14  title                  41184 non-null  str    
 15  video             

In [17]:
display(f"Número de linhas após limpeza: {len(df_movies_limpo)}")
display(f"Número de linhas removidas: {len(df_movies) - len(df_movies_limpo)}")

'Número de linhas após limpeza: 41184'

'Número de linhas removidas: 4282'

---
## Salvando Dataset Limpo

In [18]:
# Salvando o DataFrame processado para um novo CSV
df_movies_limpo.to_csv('../data/processed/movies_cleaned.csv', index=False)

---
## Conclusão da Limpeza e Estruturação de Dados

O dataset de filmes passou pelas etapas essenciais de higienização e formatação para garantir a integridade das análises financeiras e de recepção do público. As principais intervenções nesta etapa incluíram:
* **Descarte de colunas de alta cardinalidade de texto**
    - Colunas descartadas: `homepage`, `poster_path`, `overview`, `tagline`, `status` e `imdb_id`.

# Pendentes:
* **Tratamento de Inconsistências:** Identificação e manejo de valores nulos e registros discrepantes, com atenção especial às métricas críticas de negócio, como `budget` (orçamento) e `revenue` (receita). 
* **Padronização e Tipagem (Data Casting):** Conversão de formatos genéricos para tipos otimizados. Isso incluiu a transformação de indicadores binários para booleanos e a estruturação de datas de lançamento para `datetime`, habilitando futuras análises de séries temporais.
# Fim dos Pendentes 

**Exportação:**
O dataset resultante, agora estruturalmente validado, foi exportado com sucesso para `data/processed/movies_cleaned.csv`.

**Próximo Passo:**
Com a base de filmes íntegra e padronizada, o pipeline avança para a **Engenharia de Features** (para criação de variáveis como lucro e extração de dicionários JSON) e, na sequência, para a **Análise Exploratória de Dados (EDA)**, onde investigaremos as dinâmicas de bilheteria e engajamento.

* **Vá para o notebook:** `08_feature_engineering_movies.ipynb`.
---